In [1]:
import json
import subprocess

In [2]:
def pandoc_input_formats():
  pandoc = subprocess.run(
      ["pandoc", "--list-input-formats"],
      capture_output=True,
      encoding="utf8"
  )
  return [fmt for fmt in pandoc.stdout.split("\n") if fmt]

pandoc_input_formats()

['biblatex',
 'bibtex',
 'bits',
 'commonmark',
 'commonmark_x',
 'creole',
 'csljson',
 'csv',
 'djot',
 'docbook',
 'docx',
 'dokuwiki',
 'endnotexml',
 'epub',
 'fb2',
 'gfm',
 'haddock',
 'html',
 'ipynb',
 'jats',
 'jira',
 'json',
 'latex',
 'man',
 'markdown',
 'markdown_github',
 'markdown_mmd',
 'markdown_phpextra',
 'markdown_strict',
 'mdoc',
 'mediawiki',
 'muse',
 'native',
 'odt',
 'opml',
 'org',
 'pod',
 'ris',
 'rst',
 'rtf',
 't2t',
 'textile',
 'tikiwiki',
 'tsv',
 'twiki',
 'typst',
 'vimwiki',
 'xml']

In [6]:
from imagewriter.document import (
    Attr, Header, Para, Str, Space, Emph, Strong, Code, Link,
    Target, HorizontalRule, BlockQuote, SoftBreak, CodeBlock,
    BulletList, Plain, OrderedList, Table, Caption, Cell, Row,
    TableHead, TableBody, TableFoot, ListAttributes, ListNumberStyle, ListNumberDelim,
    Alignment, ColSpec
)

with open("./tests/documents/test.md", "r") as f:
    data = f.read()
pandoc = subprocess.run(
    ["pandoc", "-r", "markdown", "-w", "json"],
    input=data,
    capture_output=True,
    encoding="utf8"
)
output = json.loads(pandoc.stdout)


def to_attr(contents):
    return Attr(
        contents[0],
        contents[1],
        [ (t[0], t[1]) for t in contents[2] ]
    )

def to_target(contents):
    return Target(contents[0], contents[1])

def to_inline(contents):
    inlines = []

    for content in contents:
        type_ = content["t"]
        cont = content.get("c", None)
        if type_ == "Str":
            inlines.append(Str(cont))
        elif type_ == "Space":
            inlines.append(Space())
        elif type_ == "Emph":
            inlines.append(Emph(to_inline(cont)))
        elif type_ == "Strong":
            inlines.append(Strong(to_inline(cont)))
        elif type_ == "Code":
            inlines.append(Code(
                to_attr(cont[0]),
                cont[1]
            ))
        elif type_ == "Link":
            inlines.append(Link(
                to_attr(cont[0]),
                to_inline(cont[1]),
                to_target(cont[2])
            ))
        elif type_ == "SoftBreak":
            inlines.append(SoftBreak())
        else:
            inlines.append(content)
    
    return inlines


def to_header(contents):
    return Header(
        contents[0],
        to_attr(contents[1]),
        to_inline(contents[2])
    )


def to_para(contents):
    return Para(to_inline(contents))


def to_block_quote(contents):
    return BlockQuote([
        to_block(block)
        for block in contents
    ])


def to_code_block(contents):
    return CodeBlock(
        to_attr(contents[0]),
        contents[1]
    )

# These should probably be literal string types
LIST_NUMBER_STYLES = {
    enum.value: enum
    for enum in ListNumberStyle
}

LIST_NUMBER_DELIMS = {
    enum.value: enum
    for enum in ListNumberDelim
}


def to_list_attributes(contents):
    return ListAttributes(
        contents[0],
        LIST_NUMBER_STYLES[contents[1]["t"]],
        LIST_NUMBER_DELIMS[contents[2]["t"]]
    )


def to_bullet_list(contents):
    return BulletList(
        [
            [to_block(block) for block in item]
            for item in contents
        ]
    )

def to_ordered_list(contents):
    return OrderedList(
        to_list_attributes(contents[0]),
        [
            [to_block(block) for block in item]
            for item in contents[1]
        ]
    )


def to_caption(contents):
    return Caption(
        to_inline(contents[0]) if contents[0] is not None else None,
        [to_block(block) for block in contents[1]]
    )


# Thos should probably be a string literal type
ALIGNMENTS = {
    enum.value: enum
    for enum in Alignment
}

def to_cell(contents):
    return Cell(
        to_attr(contents[0]),
        ALIGNMENTS[contents[1]["t"]],
        contents[2],
        contents[3],
        [
            to_block(block)
            for block in contents[4]
        ]       
    )


def to_row(contents):
    return Row(
        to_attr(contents[0]),
        [ to_cell(cell) for cell in contents[1] ]
    )


# Column width should probably be typed more like Caption
def to_col_width(width):
    type_ = width["t"]
    value = None
    if type_ == "ColWidth":
        value = width["c"]
    return value
        
    
# TODO
def to_col_specs(contents):
    return [
        ColSpec(
            ALIGNMENTS[spec[0]["t"]],
            to_col_width(spec[1])
        )
        for spec in contents
    ]


def to_table_header(contents):
    return TableHead(
        to_attr(contents[0]),
        [
            to_row(row) for row in contents[1]
        ]
    )


def to_table_body(contents):
    return TableBody(
        to_attr(contents[0]),
        contents[1],
        [ to_row(row) for row in contents[2]],
        [ to_row(row) for row in contents[3]]
    )

def to_table_footer(contents):
    return TableFoot(
        to_attr(contents[0]),
        [ to_row(row) for row in contents[1]]
    )


def to_table(contents):
    # return contents
    return Table(
        to_attr(contents[0]),
        to_caption(contents[1]),
        to_col_specs(contents[2]),
        to_table_header(contents[3]),
        [to_table_body(body) for body in contents[4]],
        to_table_footer(contents[5])
    )
    


def to_block(block):
    type_ = block["t"]
    contents = block.get("c", None)
    if type_ == "Header":
        return to_header(contents)
    if type_ == "Para":
        return to_para(contents)
    if type_ == "HorizontalRule":
        return HorizontalRule()
    if type_ == "BlockQuote":
        return to_block_quote(contents)
    if type_ == "CodeBlock":
        return to_code_block(contents)
    if type_ == "BulletList":
        return to_bullet_list(contents)
    if type_ == "Plain":
        return Plain(to_inline(contents))
    if type_ == "OrderedList":
        return to_ordered_list(contents)
    if type_ == "Table":
        return to_table(contents)
    return block


doc = [
    to_block(block)
    for block in output["blocks"]
]

from pprint import pprint

for block in doc:
    pprint(block)
    print('---')

Header(level=1,
       attr=Attr(identifier='header-1', classes=[], pairs=[]),
       contents=[Str(contents='Header'), Space(), Str(contents='1')])
---
Header(level=2,
       attr=Attr(identifier='header-2', classes=[], pairs=[]),
       contents=[Str(contents='Header'), Space(), Str(contents='2')])
---
Header(level=3,
       attr=Attr(identifier='header-3', classes=[], pairs=[]),
       contents=[Str(contents='Header'), Space(), Str(contents='3')])
---
Header(level=4,
       attr=Attr(identifier='header-4', classes=[], pairs=[]),
       contents=[Str(contents='Header'), Space(), Str(contents='4')])
---
Para(contents=[Str(contents='A'),
               Space(),
               Str(contents='paragraph'),
               Space(),
               Str(contents='with'),
               Space(),
               Emph(contents=[Str(contents='emphasized')]),
               Space(),
               Str(contents='and'),
               Space(),
               Strong(contents=[Str(contents='bold')]),
   